# 🔧 Vietnamese Auto Parts – Sales Quantity Forecasting (v3)

**Task**: Predict 56 days of daily sales qty across ~15,972 SKUs  
**Metric**: WRMSSE (Weighted Root Mean Squared Scaled Error)  
**Key improvements vs v2**:
- Proper return-transaction handling (net daily qty)  
- Formal time-based Train / Val / Test split  
- WRMSSE evaluation on held-out windows  
- Intermittent-demand (Croston) for sparse SKUs  
- XGBoost added to ensemble  
- Zero-inflation post-processing  
- Vietnamese holiday calendar features  


In [34]:
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from datetime import timedelta
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

SEED = 42
np.random.seed(SEED)

ROOT = Path(os.getcwd()).resolve()
DATA_DIR  = ROOT / "data"
OUT_DIR   = ROOT / "output"
OUT_DIR.mkdir(exist_ok=True)

TRAIN_PATH  = DATA_DIR / "train.csv"
OUTPUT_PATH = OUT_DIR / "submission_v3.csv"


# ── Key dates ───────────────────────────────────────────────────
LAST_TRAIN  = pd.Timestamp("2025-09-05")   # last day in train.csv
# Validation window (mimics public leaderboard): 28 days before last_train - 28
VAL_START   = pd.Timestamp("2025-07-12")   # 56 days before last_train+1
VAL_END     = pd.Timestamp("2025-08-08")   # +28 days
TEST_START  = pd.Timestamp("2025-08-09")   # next 28 days
TEST_END    = pd.Timestamp("2025-09-05")   # = last training day

CUTOFF_TRAIN = VAL_START - timedelta(days=1)  # 2025-07-11

# Actual forecast horizon
FORECAST_DATES = pd.date_range("2025-09-06", periods=56, freq="D")
N_FORECAST     = 56

print(f"Training up to : {CUTOFF_TRAIN.date()}")
print(f"Val window     : {VAL_START.date()} → {VAL_END.date()} (28 days)")
print(f"Test window    : {TEST_START.date()} → {TEST_END.date()} (28 days)")
print(f"Forecast window: {FORECAST_DATES[0].date()} → {FORECAST_DATES[-1].date()} (56 days)")

Training up to : 2025-07-11
Val window     : 2025-07-12 → 2025-08-08 (28 days)
Test window    : 2025-08-09 → 2025-09-05 (28 days)
Forecast window: 2025-09-06 → 2025-10-31 (56 days)


## 1. Vietnamese Holiday Calendar

In [35]:
# Key recurring Vietnamese holidays (approximate Gregorian dates)
# Lunar-based holidays shift year to year — we include the main ones
VIET_HOLIDAYS = set()

for year in range(2020, 2026):
    # Fixed Gregorian
    VIET_HOLIDAYS.update([
        pd.Timestamp(f"{year}-01-01"),   # New Year
        pd.Timestamp(f"{year}-04-30"),   # Reunification Day
        pd.Timestamp(f"{year}-05-01"),   # Labour Day
        pd.Timestamp(f"{year}-09-02"),   # National Day
    ])

# Lunar Tet (approx) – major demand shock for auto parts
TET_DATES = {
    2021: ("2021-02-10","2021-02-16"),
    2022: ("2022-01-31","2022-02-06"),
    2023: ("2023-01-21","2023-01-27"),
    2024: ("2024-02-08","2024-02-14"),
    2025: ("2025-01-28","2025-02-03"),
}
for yr, (s, e) in TET_DATES.items():
    for d in pd.date_range(s, e):
        VIET_HOLIDAYS.add(d)

print(f"Total holiday/Tet dates tracked: {len(VIET_HOLIDAYS)}")

Total holiday/Tet dates tracked: 59


## 2. Load & Clean Data

In [36]:
print("[1/8] Loading raw data …")
raw = pd.read_csv(TRAIN_PATH, low_memory=False)
raw["Date"] = pd.to_datetime(raw["Date"])

# Numeric coercion (commas in some fields)
for c in ["Quantity", "SalesAmount", "Cost Amount"]:
    if c in raw.columns:
        raw[c] = pd.to_numeric(
            raw[c].astype(str).str.replace(",", "."), errors="coerce"
        ).fillna(0)

print(f"  Raw shape : {raw.shape}")
print(f"  Date range: {raw.Date.min().date()} → {raw.Date.max().date()}")
print(f"  SKUs      : {raw.ItemCode.nunique():,}")
print(f"  Neg qty rows: {(raw.Quantity < 0).sum():,}")

# ── Return-transaction handling ────────────────────────────────
# Strategy: compute NET daily qty per SKU (sales minus returns).
# The net may be negative on heavy-return days; we clip to 0 for the
# training target but preserve the information via a "return_ratio" feature.

daily_net = (
    raw.groupby(["ItemCode", "Date"])["Quantity"]
    .sum()
    .reset_index()
    .rename(columns={"Quantity": "qty_net"})
)

# Also track gross returns separately (useful for features)
daily_ret = (
    raw[raw.Quantity < 0]
    .groupby(["ItemCode", "Date"])["Quantity"]
    .sum()
    .abs()
    .reset_index()
    .rename(columns={"Quantity": "qty_returned"})
)

daily = daily_net.merge(daily_ret, on=["ItemCode","Date"], how="left")
daily["qty_returned"] = daily["qty_returned"].fillna(0)
# Target: clip net to 0 (non-negative quantity for forecasting)
daily["qty"] = daily["qty_net"].clip(lower=0)

all_skus = sorted(daily["ItemCode"].unique().tolist())
N_SKUS   = len(all_skus)
print(f"\n  Daily rows after netting: {len(daily):,}")
print(f"  Days clipped to 0 from negative net: {(daily.qty_net < 0).sum():,}")
print(f"  All SKUs: {N_SKUS:,}")

[1/8] Loading raw data …


  Raw shape : (711980, 8)
  Date range: 2020-11-17 → 2025-09-05
  SKUs      : 15,972
  Neg qty rows: 37,434

  Daily rows after netting: 507,050
  Days clipped to 0 from negative net: 20,733
  All SKUs: 15,972


## 3. Exploratory Data Analysis

In [37]:
# ── 3a. SKU sales distribution ───────────────────────────────────
sku_total = daily.groupby("ItemCode")["qty"].sum().sort_values(ascending=False)
cumsum_pct = sku_total.cumsum() / sku_total.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(range(len(cumsum_pct)), cumsum_pct.values, linewidth=2)
axes[0].axhline(80, color="red", linestyle="--", label="80%")
axes[0].axhline(95, color="orange", linestyle="--", label="95%")
axes[0].set_xlabel("SKU rank"); axes[0].set_ylabel("Cumulative qty %")
axes[0].set_title("Long-tail: Cumulative Sales by SKU Rank")
axes[0].legend(); axes[0].grid(alpha=0.3)

# top 20% SKUs → what % of sales?
top20pct = int(N_SKUS * 0.20)
top20_sales = sku_total.iloc[:top20pct].sum() / sku_total.sum() * 100
axes[0].axvline(top20pct, color="green", linestyle=":", label=f"Top 20% SKUs → {top20_sales:.0f}% sales")
axes[0].legend()

# ── 3b. Daily total qty over time ────────────────────────────────
daily_agg = daily.groupby("Date")["qty"].sum().reset_index()
daily_agg["roll7"] = daily_agg["qty"].rolling(7).mean()
axes[1].plot(daily_agg.Date, daily_agg.qty, alpha=0.3, color="steelblue", label="Daily")
axes[1].plot(daily_agg.Date, daily_agg.roll7, color="steelblue", linewidth=2, label="7d MA")
for h in VIET_HOLIDAYS:
    if daily_agg.Date.min() <= h <= daily_agg.Date.max():
        axes[1].axvline(h, color="red", alpha=0.15, linewidth=0.8)
# Shade val/test windows
axes[1].axvspan(VAL_START, VAL_END, alpha=0.15, color="orange", label="Val window")
axes[1].axvspan(TEST_START, TEST_END, alpha=0.15, color="green", label="Test window")
axes[1].set_title("Total Daily Qty Over Time")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.savefig(OUT_DIR / "eda_overview.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved eda_overview.png")

# Return rate
total_gross = raw.groupby("ItemCode")["Quantity"].apply(lambda x: x[x>0].sum())
total_ret   = raw.groupby("ItemCode")["Quantity"].apply(lambda x: x[x<0].abs().sum())
ret_rate = (total_ret / total_gross.clip(lower=1)).describe()
print("\nReturn rate per SKU:", ret_rate.to_string())

Saved eda_overview.png

Return rate per SKU: count    15972.000000
mean         0.101449
std          0.206708
min          0.000000
25%          0.000000
50%          0.001808
75%          0.100000
max          1.800000


## 4. SKU Segmentation & Statistics

In [38]:
print("[2/8] SKU segmentation …")

agg = daily.groupby("ItemCode").agg(
    active_days   = ("qty", "count"),
    total_qty     = ("qty", "sum"),
    mean_qty      = ("qty", "mean"),
    median_qty    = ("qty", "median"),
    std_qty       = ("qty", "std"),
    last_date     = ("Date", "max"),
    zero_rate     = ("qty", lambda x: (x == 0).mean()),
).reset_index().fillna({"std_qty": 0, "zero_rate": 0})

# Recent demand (last 90 / 180 days of TRAINING data)
for d, col in [(90, "r90"), (180, "r180"), (365, "r365")]:
    cut = CUTOFF_TRAIN - timedelta(days=d)
    r = (daily[(daily.Date >= cut) & (daily.Date <= CUTOFF_TRAIN)]
         .groupby("ItemCode")["qty"].sum().reset_index()
         .rename(columns={"qty": col}))
    agg = agg.merge(r, on="ItemCode", how="left").fillna({col: 0})

# Coefficient of variation
agg["cv"] = (agg["std_qty"] / agg["mean_qty"].clip(lower=1e-6)).clip(0, 10)

# ── Tiering ──────────────────────────────────────────────────────
# Tier 0: dead (≤5 active days)   → predict 0
# Tier 1: very sparse (≤20)       → simple moving average
# Tier 2: sparse (≤50)            → Croston / seasonal
# Tier 3: regular (>50)           → LightGBM + XGBoost ensemble
conds = [
    (agg.active_days <= 5),
    (agg.active_days <= 20),
    (agg.active_days <= 50),
]
agg["tier"] = np.select(conds, [0, 1, 2], default=3)

for t in range(4):
    s = agg[agg.tier == t]
    v = s.total_qty.sum()
    pct_sku = 100 * len(s) / N_SKUS
    pct_qty = 100 * v / agg.total_qty.sum()
    print(f"  Tier {t}: {len(s):5d} SKUs ({pct_sku:.1f}%) │ "
          f"{v:>10,.0f} qty ({pct_qty:.1f}%)")

tier3_skus = agg[agg.tier == 3]["ItemCode"].tolist()
sku2idx = {s: i for i, s in enumerate(tier3_skus)}
N_T3 = len(tier3_skus)
print(f"\n  Tier-3 SKUs for ML model: {N_T3:,}")

[2/8] SKU segmentation …
  Tier 0:  7492 SKUs (46.9%) │     35,946 qty (1.4%)
  Tier 1:  4126 SKUs (25.8%) │    119,568 qty (4.8%)
  Tier 2:  2050 SKUs (12.8%) │    183,761 qty (7.4%)
  Tier 3:  2304 SKUs (14.4%) │  2,154,389 qty (86.4%)

  Tier-3 SKUs for ML model: 2,304


## 5. WRMSSE Evaluation Function

In [39]:
def compute_wrmsse(
    daily_df: pd.DataFrame,
    forecast_dict: dict,
    eval_start: pd.Timestamp,
    eval_end: pd.Timestamp,
    train_end: pd.Timestamp,
    all_sku_list: list,
    weight_col: str = "qty",
) -> dict:
    """
    Compute Weighted RMSSE (WRMSSE) for a forecast vs actuals.

    Parameters
    ----------
    daily_df    : long-format daily dataframe with cols [ItemCode, Date, qty]
    forecast_dict : {ItemCode: np.array of shape (n_days,)} – aligned to eval window
    eval_start  : first day of evaluation window
    eval_end    : last day of evaluation window
    train_end   : last day of training data (for scaling denominator)
    all_sku_list: list of all SKUs in submission order
    weight_col  : column used for weighting (default = qty)

    Returns
    -------
    dict with keys: wrmsse, per_sku_rmsse (Series), weights (Series)
    """
    h = (eval_end - eval_start).days + 1   # forecast horizon
    eval_dates = pd.date_range(eval_start, eval_end, freq="D")

    # ── Actuals matrix ──────────────────────────────────────────
    act = (daily_df[(daily_df.Date >= eval_start) & (daily_df.Date <= eval_end)]
           .pivot_table(index="ItemCode", columns="Date", values="qty", aggfunc="sum")
           .reindex(columns=eval_dates)
           .fillna(0))

    # ── Denominator: naive-1 MSE over training period ──────────
    train_sub = daily_df[daily_df.Date <= train_end].copy()
    # Full calendar (fill missing dates with 0 for each SKU)
    all_dates_train = pd.date_range(train_sub.Date.min(), train_end, freq="D")
    pivot_train = (train_sub
                   .pivot_table(index="ItemCode", columns="Date", values="qty", aggfunc="sum")
                   .reindex(columns=all_dates_train)
                   .fillna(0))

    # naive-1 squared errors per SKU: mean of (y_t - y_{t-1})^2
    diffs = pivot_train.diff(axis=1).iloc[:, 1:]   # drop first NaN col
    naive_mse = (diffs ** 2).mean(axis=1).clip(lower=1e-6)   # avoid /0

    # ── RMSSE per SKU ────────────────────────────────────────────
    rmsse_list = []
    for sku in all_sku_list:
        y_hat = np.array(forecast_dict.get(sku, np.zeros(h)), dtype=float)[:h]
        if sku in act.index:
            y_true = act.loc[sku].values.astype(float)
        else:
            y_true = np.zeros(h)

        mse_f = np.mean((y_hat - y_true) ** 2)
        denom = naive_mse.get(sku, 1.0)
        rmsse_list.append(np.sqrt(mse_f / denom))

    per_sku_rmsse = pd.Series(rmsse_list, index=all_sku_list)

    # ── Weights = share of total train-period sales ───────────
    sku_weights_raw = (daily_df[daily_df.Date <= train_end]
                       .groupby("ItemCode")[weight_col].sum()
                       .reindex(all_sku_list)
                       .fillna(0))
    total_w = sku_weights_raw.sum()
    weights = sku_weights_raw / (total_w if total_w > 0 else 1.0)

    wrmsse = float((weights * per_sku_rmsse).sum())
    return {
        "wrmsse": wrmsse,
        "per_sku_rmsse": per_sku_rmsse,
        "weights": weights,
    }

print("✓ WRMSSE function defined")
print("  Usage: compute_wrmsse(daily, forecast_dict, eval_start, eval_end, train_end, all_skus)")

✓ WRMSSE function defined
  Usage: compute_wrmsse(daily, forecast_dict, eval_start, eval_end, train_end, all_skus)


## 6. Time-Based Train / Val / Test Split

In [40]:
# ── Splits ───────────────────────────────────────────────────────
# For local CV we use the FULL dataset as a proxy, making held-out
# windows that mirror the real submission windows.
#
#  |── train_cv ──────────────────|── val (28d) ──|── test (28d) ──|
#                                 VAL_START        TEST_START       LAST_TRAIN
#
daily_train_cv = daily[daily.Date <= CUTOFF_TRAIN].copy()
daily_val      = daily[(daily.Date >= VAL_START) & (daily.Date <= VAL_END)].copy()
daily_test     = daily[(daily.Date >= TEST_START) & (daily.Date <= TEST_END)].copy()
# Full training (for final submission model — uses all data up to last train date)
daily_full     = daily[daily.Date <= LAST_TRAIN].copy()

print(f"Train-CV rows  : {len(daily_train_cv):,}  (up to {CUTOFF_TRAIN.date()})")
print(f"Val rows       : {len(daily_val):,}  ({VAL_START.date()} → {VAL_END.date()})")
print(f"Test rows      : {len(daily_test):,}  ({TEST_START.date()} → {TEST_END.date()})")
print(f"Full-train rows: {len(daily_full):,}  (for final submission)")

# Recompute SKU stats on training data only (no leakage)
def compute_sku_stats(df_train, last_day):
    """Compute per-SKU statistics from training data only."""
    s = df_train.groupby("ItemCode").agg(
        active_days = ("qty", "count"),
        total_qty   = ("qty", "sum"),
        mean_qty    = ("qty", "mean"),
        median_qty  = ("qty", "median"),
        std_qty     = ("qty", "std"),
        zero_rate   = ("qty", lambda x: (x == 0).mean()),
    ).reset_index().fillna({"std_qty": 0})
    s["cv"] = (s["std_qty"] / s["mean_qty"].clip(lower=1e-6)).clip(0, 10)

    for d, col in [(90, "r90"), (180, "r180")]:
        cut = last_day - timedelta(days=d)
        r = (df_train[df_train.Date >= cut].groupby("ItemCode")["qty"]
             .sum().reset_index().rename(columns={"qty": col}))
        s = s.merge(r, on="ItemCode", how="left").fillna({col: 0})
    return s

sku_stats_cv   = compute_sku_stats(daily_train_cv, CUTOFF_TRAIN)
sku_stats_full = compute_sku_stats(daily_full, LAST_TRAIN)
print("\nSKU stats computed for CV and full-train sets.")

Train-CV rows  : 490,588  (up to 2025-07-11)
Val rows       : 8,375  (2025-07-12 → 2025-08-08)
Test rows      : 8,087  (2025-08-09 → 2025-09-05)
Full-train rows: 507,050  (for final submission)

SKU stats computed for CV and full-train sets.


## 7. Dense Matrix Feature Engineering

In [41]:
def build_dense_matrix(daily_df, skus, forecast_dates, hist_start=None):
    """Build (N_SKU × N_DATES_ALL) matrix and helpers."""
    if hist_start is None:
        hist_start = daily_df.Date.min()

    all_dates = pd.date_range(hist_start, forecast_dates[-1], freq="D")
    n_hist    = (daily_df.Date.max() - hist_start).days + 1

    sku2i  = {s: i for i, s in enumerate(skus)}
    date2i = {d: i for i, d in enumerate(all_dates)}
    n_skus = len(skus)
    n_dates = len(all_dates)

    M = np.zeros((n_skus, n_dates), dtype=np.float32)

    sub = daily_df[daily_df.ItemCode.isin(skus)]
    for row in sub.itertuples(index=False):
        si = sku2i.get(row.ItemCode)
        di = date2i.get(row.Date)
        if si is not None and di is not None:
            M[si, di] = row.qty

    return M, all_dates, date2i, n_hist


HIST_START_CV   = CUTOFF_TRAIN  - timedelta(days=729)
HIST_START_FULL = LAST_TRAIN    - timedelta(days=729)

# CV matrix (training data only → tier3 skus need stats from cv)
cv_tier3_skus  = (sku_stats_cv[sku_stats_cv.active_days > 50]["ItemCode"].tolist())
sku_stats_cv3  = (sku_stats_cv[sku_stats_cv.ItemCode.isin(cv_tier3_skus)]
                  .set_index("ItemCode").reindex(cv_tier3_skus).reset_index())

print("[3/8] Building CV dense matrix …")
VAL_DATES = pd.date_range(VAL_START, TEST_END)   # 56 days for CV model
M_cv, all_dates_cv, date2i_cv, hist_len_cv = build_dense_matrix(
    daily_train_cv, cv_tier3_skus, VAL_DATES, HIST_START_CV)
print(f"  CV Matrix: {M_cv.shape} | Non-zero: {np.count_nonzero(M_cv):,}")

# Full matrix (all training data)
full_tier3_skus = (sku_stats_full[sku_stats_full.active_days > 50]["ItemCode"].tolist())
sku_stats_full3 = (sku_stats_full[sku_stats_full.ItemCode.isin(full_tier3_skus)]
                   .set_index("ItemCode").reindex(full_tier3_skus).reset_index())

print("[3b/8] Building Full dense matrix …")
M_full, all_dates_full, date2i_full, hist_len_full = build_dense_matrix(
    daily_full, full_tier3_skus, FORECAST_DATES, HIST_START_FULL)
print(f"  Full Matrix: {M_full.shape} | Non-zero: {np.count_nonzero(M_full):,}")

[3/8] Building CV dense matrix …
  CV Matrix: (2253, 786) | Non-zero: 183,529
[3b/8] Building Full dense matrix …
  Full Matrix: (2304, 786) | Non-zero: 180,518


## 8. Feature Engineering (Vectorised)

In [42]:
# Vietnamese holidays as integer set for fast lookup
HOLIDAY_SET = {d.toordinal() for d in VIET_HOLIDAYS}

def is_near_holiday(d, window=3):
    ord_d = d.toordinal()
    return int(any(abs(ord_d - h) <= window for h in HOLIDAY_SET))

def days_to_next_holiday(d, horizon=30):
    ord_d = d.toordinal()
    gaps = [h - ord_d for h in HOLIDAY_SET if 0 < h - ord_d <= horizon]
    return min(gaps) if gaps else horizon

def compute_features(M, col_idx, date, sku_stats_df):
    """Return feature matrix X of shape (N_SKU, n_features)."""
    n = M.shape[0]
    parts = []

    # ─── Lags ────────────────────────────────────────────────────
    for lag in [1, 2, 3, 7, 14, 21, 28, 35, 42, 56, 91, 182, 364]:
        li = col_idx - lag
        parts.append(M[:, li] if li >= 0 else np.zeros(n))

    # ─── Rolling mean ────────────────────────────────────────────
    for w in [3, 7, 14, 28, 56, 91, 182]:
        lo = max(0, col_idx - w)
        parts.append(M[:, lo:col_idx].mean(axis=1))

    # ─── Rolling std / max ───────────────────────────────────────
    for w in [7, 28, 91]:
        lo = max(0, col_idx - w)
        seg = M[:, lo:col_idx]
        parts.append(seg.std(axis=1))
        parts.append(seg.max(axis=1))

    # ─── Zero-rate (proportion of zeros in last 28/91 days) ──────
    for w in [28, 91]:
        lo = max(0, col_idx - w)
        seg = M[:, lo:col_idx]
        parts.append((seg == 0).mean(axis=1))

    # ─── EWM approximation ───────────────────────────────────────
    for span in [7, 28, 91]:
        alpha = 2.0 / (span + 1)
        lo = max(0, col_idx - span * 3)
        seg = M[:, lo:col_idx]
        if seg.shape[1] == 0:
            parts.append(np.zeros(n))
        else:
            w_idx   = np.arange(seg.shape[1], dtype=np.float32)
            weights = (1 - alpha) ** (seg.shape[1] - 1 - w_idx)
            weights /= weights.sum()
            parts.append((seg * weights[None, :]).sum(axis=1))

    # ─── Trend: ratio recent 28d mean / overall mean ─────────────
    lo28 = max(0, col_idx - 28)
    rm28 = M[:, lo28:col_idx].mean(axis=1)
    mean_q = sku_stats_df["mean_qty"].values.astype(np.float32)
    trend  = rm28 / np.where(mean_q > 0, mean_q, 1.0)
    parts.append(trend.clip(0, 20))

    # YoY: value 364 days ago vs recent mean
    yoy_li = col_idx - 364
    yoy_val = M[:, yoy_li] if yoy_li >= 0 else np.zeros(n)
    yoy_ratio = yoy_val / np.where(mean_q > 0, mean_q, 1.0)
    parts.append(yoy_ratio.clip(0, 20))

    # ─── Calendar ────────────────────────────────────────────────
    iso_week = date.isocalendar()[1]
    parts.append(np.full(n, date.dayofweek))
    parts.append(np.full(n, date.month))
    parts.append(np.full(n, iso_week))
    parts.append(np.full(n, date.dayofyear))
    parts.append(np.full(n, date.year - 2020))       # relative year
    parts.append(np.full(n, int(date.dayofweek >= 5)))
    parts.append(np.full(n, date.quarter))
    parts.append(np.full(n, int(date.is_month_end)))
    parts.append(np.full(n, int(date.is_month_start)))
    parts.append(np.full(n, date.day))
    # Holiday features
    parts.append(np.full(n, int(date in VIET_HOLIDAYS)))
    parts.append(np.full(n, is_near_holiday(date, 3)))
    parts.append(np.full(n, days_to_next_holiday(date, 30)))
    # sin/cos encoding for month and day-of-week
    parts.append(np.full(n, np.sin(2 * np.pi * date.month / 12)))
    parts.append(np.full(n, np.cos(2 * np.pi * date.month / 12)))
    parts.append(np.full(n, np.sin(2 * np.pi * date.dayofweek / 7)))
    parts.append(np.full(n, np.cos(2 * np.pi * date.dayofweek / 7)))

    # ─── SKU-level stats ─────────────────────────────────────────
    for col in ["mean_qty", "median_qty", "std_qty", "cv",
                "active_days", "zero_rate", "r90", "r180"]:
        parts.append(sku_stats_df[col].values.astype(np.float32))

    X = np.column_stack(parts)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# Feature count
_n_feat = compute_features(
    np.zeros((5, 400)), 300, pd.Timestamp("2024-01-15"),
    pd.DataFrame({c: np.zeros(5) for c in
                  ["mean_qty","median_qty","std_qty","cv",
                   "active_days","zero_rate","r90","r180"]})
).shape[1]
print(f"✓ Feature matrix per day: {_n_feat} features per SKU")

✓ Feature matrix per day: 58 features per SKU


## 9. Build Training Dataset

In [43]:
def build_train_dataset(M, all_dates, hist_len, sku_stats_df,
                        sample_every=3, skip_first=370):
    """Sample date-columns from history window to build (X, y)."""
    sample_cols = list(range(skip_first, hist_len, sample_every))
    print(f"  Sampling {len(sample_cols)} date-cols × {M.shape[0]} SKUs = "
          f"{len(sample_cols)*M.shape[0]:,} rows")
    X_parts, y_parts = [], []
    for ci in sample_cols:
        d = all_dates[ci]
        X = compute_features(M, ci, d, sku_stats_df)
        y = M[:, ci]
        X_parts.append(X)
        y_parts.append(y)
    return np.vstack(X_parts), np.concatenate(y_parts)

def build_val_dataset(M, all_dates, date2i, val_start, val_end, sku_stats_df):
    """Build validation (X, y) for a contiguous date range."""
    val_dates = pd.date_range(val_start, val_end, freq="D")
    X_parts, y_parts = [], []
    for d in val_dates:
        ci = date2i.get(d)
        if ci is None: continue
        X = compute_features(M, ci, d, sku_stats_df)
        y = M[:, ci]
        X_parts.append(X)
        y_parts.append(y)
    return np.vstack(X_parts), np.concatenate(y_parts)

print("[4/8] Building CV training data …")
X_tr, y_tr = build_train_dataset(M_cv, all_dates_cv, hist_len_cv, sku_stats_cv3, sample_every=3)
print(f"  X_train: {X_tr.shape}")

print("[4b/8] Building CV validation data …")
X_val, y_val = build_val_dataset(M_cv, all_dates_cv, date2i_cv, VAL_START, VAL_END, sku_stats_cv3)
print(f"  X_val  : {X_val.shape}")

[4/8] Building CV training data …
  Sampling 120 date-cols × 2253 SKUs = 270,360 rows
  X_train: (270360, 58)
[4b/8] Building CV validation data …
  X_val  : (63084, 58)


## 10. Model Training — LightGBM + XGBoost

In [44]:
# ── LightGBM ─────────────────────────────────────────────────────
print("[5/8] Training LightGBM (CV) …")
lgb_model_cv = lgb.LGBMRegressor(
    objective       = "tweedie",
    tweedie_variance_power = 1.5,
    n_estimators    = 4000,
    learning_rate   = 0.025,
    num_leaves      = 127,
    min_child_samples = 30,
    feature_fraction  = 0.70,
    bagging_fraction  = 0.80,
    bagging_freq      = 5,
    lambda_l1         = 0.1,
    lambda_l2         = 0.2,
    n_jobs  = -1,
    verbose = -1,
    random_state = SEED,
)
lgb_model_cv.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(200, verbose=False),
        lgb.log_evaluation(500),
    ],
)
print(f"  Best iteration: {lgb_model_cv.best_iteration_}")

# ── XGBoost ──────────────────────────────────────────────────────
print("\n[5b/8] Training XGBoost (CV) …")
xgb_model_cv = xgb.XGBRegressor(
    objective    = "reg:tweedie",
    tweedie_variance_power = 1.5,
    n_estimators = 2000,
    learning_rate = 0.03,
    max_depth     = 7,
    min_child_weight = 30,
    subsample     = 0.80,
    colsample_bytree = 0.70,
    reg_alpha     = 0.1,
    reg_lambda    = 0.2,
    n_jobs        = -1,
    verbosity     = 0,
    random_state  = SEED,
    early_stopping_rounds = 150,
    eval_metric   = "rmse",
)
xgb_model_cv.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=500,
)
print(f"  Best iteration: {xgb_model_cv.best_iteration}")

[5/8] Training LightGBM (CV) …
[500]	valid_0's tweedie: 0.455268
[1000]	valid_0's tweedie: 0.37965
[1500]	valid_0's tweedie: 0.331107
[2000]	valid_0's tweedie: 0.291511
[2500]	valid_0's tweedie: 0.260161
[3000]	valid_0's tweedie: 0.232476
[3500]	valid_0's tweedie: 0.208134
[4000]	valid_0's tweedie: 0.188569
  Best iteration: 4000

[5b/8] Training XGBoost (CV) …
[0]	validation_0-rmse:0.61207
[173]	validation_0-rmse:1.41060
  Best iteration: 23


## 11. Sparse SKU Predictions (Tiers 0–2)

In [45]:
def croston_forecast(series: np.ndarray, n_ahead: int, alpha: float = 0.15) -> np.ndarray:
    """
    Simplified Croston's method for intermittent demand.
    Returns a flat array of length n_ahead.
    """
    if series.sum() == 0:
        return np.zeros(n_ahead)
    # Initialise
    non_zero = series[series > 0]
    if len(non_zero) == 0:
        return np.zeros(n_ahead)
    z = float(non_zero.mean())      # demand size
    x = len(series) / len(non_zero) # inter-arrival
    for v in series:
        if v > 0:
            z = alpha * v + (1 - alpha) * z
            x = alpha * 1 + (1 - alpha) * x
    forecast_val = z / x
    return np.full(n_ahead, max(0.0, forecast_val), dtype=np.float32)


def sparse_tier_predictions(sku_stats_df, daily_df, last_train_day,
                             forecast_dates, agg_df):
    """Predict Tiers 0/1/2 SKUs."""
    n_ahead = len(forecast_dates)

    # DOW × month seasonal index
    d2 = daily_df.copy()
    d2["dow"]   = d2.Date.dt.dayofweek
    d2["month"] = d2.Date.dt.month
    gm = d2.qty.mean().clip(1e-6)
    seasonal = (d2.groupby(["dow","month"])["qty"].mean() / gm).to_dict()

    preds = {}
    sparse_rows = agg_df[agg_df.tier.isin([0, 1, 2])]

    for _, row in sparse_rows.iterrows():
        sku  = row.ItemCode
        tier = row.tier
        m90  = row.r90 / 90.0    # avg daily rate last 90 days

        if tier == 0 or m90 == 0:
            preds[sku] = np.zeros(n_ahead, dtype=np.float32)
            continue

        if tier == 1:
            # Simple constant = recent daily mean
            preds[sku] = np.full(n_ahead, m90, dtype=np.float32)
            continue

        # Tier 2: Croston + seasonal blend
        hist_series = (daily_df[daily_df.ItemCode == sku]
                       .set_index("Date")["qty"]
                       .reindex(pd.date_range(daily_df.Date.min(), last_train_day))
                       .fillna(0).values)
        c_pred = croston_forecast(hist_series, n_ahead)

        dp = []
        for fd in forecast_dates:
            key = (fd.dayofweek, fd.month)
            s   = seasonal.get(key, 1.0)
            p   = 0.4 * c_pred[0] + 0.6 * m90 * s
            dp.append(max(0.0, p))

        preds[sku] = np.array(dp, dtype=np.float32)

    return preds


print("Defining sparse tier predictor … ✓")
print("Croston's method will be used for Tier-2 SKUs")

Defining sparse tier predictor … ✓
Croston's method will be used for Tier-2 SKUs


## 12. Recursive Forecast & WRMSSE Evaluation (CV)

In [46]:
def recursive_forecast(M, all_dates, date2i, forecast_dates,
                       lgb_m, xgb_m, sku_stats_df, lgb_weight=0.6):
    """
    Autoregressively forecast `forecast_dates` for all tier-3 SKUs.
    Returns array (N_SKU, N_FORECAST).
    """
    n_sku   = M.shape[0]
    n_ahead = len(forecast_dates)
    preds   = np.zeros((n_sku, n_ahead), dtype=np.float32)

    for step, fdate in enumerate(forecast_dates):
        ci = date2i[fdate]
        X  = compute_features(M, ci, fdate, sku_stats_df)

        p_lgb = lgb_m.predict(X).clip(0).astype(np.float32)
        p_xgb = xgb_m.predict(X).clip(0).astype(np.float32)
        p_ens = lgb_weight * p_lgb + (1 - lgb_weight) * p_xgb

        preds[:, step] = p_ens
        M[:, ci] = p_ens       # write back for next lag

        if step % 7 == 0:
            print(f"  step {step+1:2d}/56 → {fdate.date()} | "
                  f"mean={p_ens.mean():.2f}, max={p_ens.max():.0f}")

    return preds


print("[6/8] CV recursive forecast (val + test window = 56 days) …")
M_cv_copy = M_cv.copy()   # avoid in-place pollution for re-runs

cv_t3_preds = recursive_forecast(
    M_cv_copy, all_dates_cv, date2i_cv,
    VAL_DATES,          # VAL_START → TEST_END (56 days)
    lgb_model_cv, xgb_model_cv, sku_stats_cv3
)
print(f"\nCV tier-3 forecast shape: {cv_t3_preds.shape}")

# Sparse tiers
print("\nCV sparse tier predictions …")
cv_sparse = sparse_tier_predictions(
    sku_stats_cv3, daily_train_cv, CUTOFF_TRAIN, VAL_DATES,
    sku_stats_cv.merge(agg[["ItemCode","tier"]], on="ItemCode")
)

# Assemble full forecast dict
cv_forecast_dict = {}
for i, sku in enumerate(cv_tier3_skus):
    cv_forecast_dict[sku] = cv_t3_preds[i]
cv_forecast_dict.update(cv_sparse)
for sku in all_skus:
    if sku not in cv_forecast_dict:
        cv_forecast_dict[sku] = np.zeros(len(VAL_DATES))

[6/8] CV recursive forecast (val + test window = 56 days) …
  step  1/56 → 2025-07-12 | mean=0.29, max=52
  step  8/56 → 2025-07-19 | mean=0.29, max=55
  step 15/56 → 2025-07-26 | mean=0.29, max=26
  step 22/56 → 2025-08-02 | mean=0.24, max=25
  step 29/56 → 2025-08-09 | mean=0.25, max=56
  step 36/56 → 2025-08-16 | mean=0.22, max=19
  step 43/56 → 2025-08-23 | mean=0.20, max=5
  step 50/56 → 2025-08-30 | mean=0.16, max=4

CV tier-3 forecast shape: (2253, 56)

CV sparse tier predictions …


In [47]:
print("\n[7/8] Evaluating WRMSSE …")

# Val WRMSSE
val_result = compute_wrmsse(
    daily_full,           # use full data for actuals
    {sku: cv_forecast_dict[sku][:28] for sku in all_skus},
    VAL_START, VAL_END,
    CUTOFF_TRAIN, all_skus
)
print(f"\n  ── Validation WRMSSE : {val_result['wrmsse']:.5f}")

# Test WRMSSE
test_result = compute_wrmsse(
    daily_full,
    {sku: cv_forecast_dict[sku][28:56] for sku in all_skus},
    TEST_START, TEST_END,
    CUTOFF_TRAIN, all_skus
)
print(f"  ── Test WRMSSE       : {test_result['wrmsse']:.5f}")

# Breakdown by tier
tier_map = agg.set_index("ItemCode")["tier"]
for t in range(4):
    t_skus = [s for s in all_skus if tier_map.get(s, 0) == t]
    w_sub  = val_result["weights"].reindex(t_skus).fillna(0)
    r_sub  = val_result["per_sku_rmsse"].reindex(t_skus).fillna(0)
    w_wrmsse = (w_sub * r_sub).sum() / w_sub.sum() if w_sub.sum() > 0 else 0
    print(f"    Tier {t} ({len(t_skus):5d} SKUs) unweighted avg RMSSE: "
          f"{r_sub.mean():.3f}  |  weighted contribution: {(w_sub*r_sub).sum():.5f}")


[7/8] Evaluating WRMSSE …

  ── Validation WRMSSE : 0.44262
  ── Test WRMSSE       : 0.42632
    Tier 0 ( 7492 SKUs) unweighted avg RMSSE: 0.338  |  weighted contribution: 0.00165
    Tier 1 ( 4126 SKUs) unweighted avg RMSSE: 0.551  |  weighted contribution: 0.01161
    Tier 2 ( 2050 SKUs) unweighted avg RMSSE: 0.826  |  weighted contribution: 0.05106
    Tier 3 ( 2304 SKUs) unweighted avg RMSSE: 0.495  |  weighted contribution: 0.37830


## 13. Feature Importance

In [48]:
feat_imp = pd.Series(
    lgb_model_cv.feature_importances_,
    index=[f"f{i}" for i in range(lgb_model_cv.n_features_in_)]
).sort_values(ascending=False).head(30)

fig, ax = plt.subplots(figsize=(10, 7))
feat_imp.plot.barh(ax=ax, color="steelblue")
ax.set_title("Top-30 Feature Importances (LightGBM CV)", fontsize=13)
ax.set_xlabel("Gain")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUT_DIR / "feat_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved feat_importance.png")

Saved feat_importance.png


## 14. Final Model — Full Training Data

In [49]:
print("[8a/8] Building full training dataset …")
X_tr_full, y_tr_full = build_train_dataset(
    M_full, all_dates_full, hist_len_full, sku_stats_full3, sample_every=3
)

# Use last 60 days of full training as early-stopping val
val_cols_full = list(range(hist_len_full - 60, hist_len_full))
Xv_full, yv_full = build_val_dataset(
    M_full, all_dates_full, date2i_full,
    LAST_TRAIN - timedelta(days=59), LAST_TRAIN, sku_stats_full3
)
print(f"  X_train_full: {X_tr_full.shape} | X_val_full: {Xv_full.shape}")

print("\n[8b/8] Training LightGBM (full) …")
lgb_model_full = lgb.LGBMRegressor(
    objective       = "tweedie",
    tweedie_variance_power = 1.5,
    n_estimators    = 5000,
    learning_rate   = 0.025,
    num_leaves      = 127,
    min_child_samples = 30,
    feature_fraction  = 0.70,
    bagging_fraction  = 0.80,
    bagging_freq      = 5,
    lambda_l1         = 0.1,
    lambda_l2         = 0.2,
    n_jobs  = -1,
    verbose = -1,
    random_state = SEED,
)
lgb_model_full.fit(
    X_tr_full, y_tr_full,
    eval_set=[(Xv_full, yv_full)],
    callbacks=[
        lgb.early_stopping(200, verbose=False),
        lgb.log_evaluation(500),
    ],
)
print(f"  Best iteration: {lgb_model_full.best_iteration_}")

print("\n[8c/8] Training XGBoost (full) …")
xgb_model_full = xgb.XGBRegressor(
    objective    = "reg:tweedie",
    tweedie_variance_power = 1.5,
    n_estimators = 3000,
    learning_rate = 0.03,
    max_depth     = 7,
    min_child_weight = 30,
    subsample     = 0.80,
    colsample_bytree = 0.70,
    reg_alpha     = 0.1,
    reg_lambda    = 0.2,
    n_jobs        = -1,
    verbosity     = 0,
    random_state  = SEED,
    early_stopping_rounds = 150,
    eval_metric   = "rmse",
)
xgb_model_full.fit(
    X_tr_full, y_tr_full,
    eval_set=[(Xv_full, yv_full)],
    verbose=500,
)
print(f"  Best iteration: {xgb_model_full.best_iteration}")

[8a/8] Building full training dataset …
  Sampling 120 date-cols × 2304 SKUs = 276,480 rows
  X_train_full: (276480, 58) | X_val_full: (138240, 58)

[8b/8] Training LightGBM (full) …
  Best iteration: 260

[8c/8] Training XGBoost (full) …
[0]	validation_0-rmse:6.75136
[500]	validation_0-rmse:5.48537
[1000]	validation_0-rmse:5.27854
[1500]	validation_0-rmse:5.19374
[2000]	validation_0-rmse:5.16073
[2500]	validation_0-rmse:5.12732
[2999]	validation_0-rmse:5.10945
  Best iteration: 2998


## 15. Final 56-Day Recursive Forecast

In [50]:
print("\nFinal recursive forecast (56 days) …")
M_full_copy = M_full.copy()

final_t3_preds = recursive_forecast(
    M_full_copy, all_dates_full, date2i_full,
    FORECAST_DATES,
    lgb_model_full, xgb_model_full, sku_stats_full3
)
print(f"\nFinal tier-3 forecast shape: {final_t3_preds.shape}")

print("\nFinal sparse tier predictions …")
final_sparse = sparse_tier_predictions(
    sku_stats_full3, daily_full, LAST_TRAIN, FORECAST_DATES,
    sku_stats_full.merge(agg[["ItemCode","tier"]], on="ItemCode", how="left")
    .assign(tier=lambda df: df.tier.fillna(0).astype(int))
)

final_all = {}
for i, sku in enumerate(full_tier3_skus):
    final_all[sku] = final_t3_preds[i]
final_all.update(final_sparse)
for sku in all_skus:
    if sku not in final_all:
        final_all[sku] = np.zeros(N_FORECAST)

print(f"\nTotal SKUs with predictions: {len(final_all):,}")


Final recursive forecast (56 days) …
  step  1/56 → 2025-09-06 | mean=0.34, max=61
  step  8/56 → 2025-09-13 | mean=0.38, max=51
  step 15/56 → 2025-09-20 | mean=0.46, max=66
  step 22/56 → 2025-09-27 | mean=0.43, max=40
  step 29/56 → 2025-10-04 | mean=0.44, max=47
  step 36/56 → 2025-10-11 | mean=0.43, max=44
  step 43/56 → 2025-10-18 | mean=0.48, max=58
  step 50/56 → 2025-10-25 | mean=0.45, max=51

Final tier-3 forecast shape: (2304, 56)

Final sparse tier predictions …

Total SKUs with predictions: 15,972


## 16. Zero-Inflation Post-Processing

In [51]:
# For SKUs with very high zero_rate, dampen small predictions
zero_rate_map = agg.set_index("ItemCode")["zero_rate"]

def apply_zero_inflation(forecast_arr, sku, threshold=0.80, dampen_below=0.5):
    """
    If a SKU's historical zero_rate > threshold,
    set forecasts < dampen_below to 0.
    """
    zr = zero_rate_map.get(sku, 0.0)
    if zr > threshold:
        forecast_arr = np.where(forecast_arr < dampen_below, 0.0, forecast_arr)
    return forecast_arr

for sku in all_skus:
    final_all[sku] = apply_zero_inflation(final_all[sku], sku)

# Sanity: no negatives
for sku in all_skus:
    final_all[sku] = np.clip(final_all[sku], 0, None)

print("✓ Zero-inflation post-processing applied")
n_nonzero = sum((final_all[s] > 0).any() for s in all_skus)
print(f"  SKUs with ≥1 non-zero forecast: {n_nonzero:,}/{N_SKUS:,}")

✓ Zero-inflation post-processing applied
  SKUs with ≥1 non-zero forecast: 4,617/15,972


## 17. Build Submission File

In [52]:
print("Building submission …")
cols = ["id"] + [f"F{i}" for i in range(1, 29)]

# ── All validation rows first, then all evaluation rows ──────────
# Matches sample_submission.csv format:
#   SKU-00001_validation, SKU-00002_validation, ...
#   SKU-00001_evaluation, SKU-00002_evaluation, ...
val_rows  = []
eval_rows = []

for sku in all_skus:
    p  = final_all[sku]
    rv = {"id": f"{sku}_validation"}
    re = {"id": f"{sku}_evaluation"}
    for i in range(28):
        rv[f"F{i+1}"] = float(max(0.0, p[i]))
        re[f"F{i+1}"] = float(max(0.0, p[28 + i]))
    val_rows.append(rv)
    eval_rows.append(re)

sub = pd.DataFrame(val_rows + eval_rows)[cols]
sub.to_csv(OUTPUT_PATH, index=False)
print(f"\n✓ Saved: {OUTPUT_PATH}")
print(f"  Rows : {len(sub):,}  | Cols : {sub.shape[1]}")
print(f"  First row id : {sub.iloc[0]['id']}")
print(f"  Middle row id: {sub.iloc[N_SKUS]['id']}  ← first evaluation row")
nz = (sub.iloc[:, 1:] > 0).any(axis=1).sum()
print(f"  Rows with any non-zero forecast: {nz:,}")

# Sanity checks
assert len(sub) == N_SKUS * 2, f"Row count mismatch: {len(sub)} vs {N_SKUS*2}"
assert not (sub.iloc[:, 1:] < 0).any().any(), "Negative values found!"
assert sub.id.nunique() == len(sub), "Duplicate IDs!"
assert all(sub.iloc[:N_SKUS]["id"].str.endswith("_validation")), "Validation block wrong!"
assert all(sub.iloc[N_SKUS:]["id"].str.endswith("_evaluation")), "Evaluation block wrong!"
print("\n✓ All sanity checks passed.")
print(sub.head(3).to_string())
print("...")
print(sub.iloc[N_SKUS:N_SKUS+3].to_string())

Building submission …

✓ Saved: D:\hbaac-2026\src\output\submission_v3.csv
  Rows : 31,944  | Cols : 29
  First row id : SKU-00001_validation
  Middle row id: SKU-00001_evaluation  ← first evaluation row
  Rows with any non-zero forecast: 9,234

✓ All sanity checks passed.
                     id        F1        F2         F3         F4         F5         F6        F7        F8        F9        F10       F11        F12        F13        F14       F15       F16        F17        F18        F19        F20        F21       F22       F23        F24        F25        F26        F27       F28
0  SKU-00001_validation  0.188889  0.188889   0.188889   0.188889   0.188889   0.188889  0.188889  0.188889  0.188889   0.188889  0.188889   0.188889   0.188889   0.188889  0.188889  0.188889   0.188889   0.188889   0.188889   0.188889   0.188889  0.188889  0.188889   0.188889   0.188889   0.188889   0.188889  0.188889
1  SKU-00002_validation  5.077936  0.000101   7.422321   7.436304   8.442800   7.206

## 18. Forecast Visualisation — Top SKUs

In [53]:
top_skus = agg.sort_values("total_qty", ascending=False).head(6)["ItemCode"].tolist()

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, sku in zip(axes, top_skus):
    hist = (daily_full[daily_full.ItemCode == sku]
            .set_index("Date")["qty"]
            .reindex(pd.date_range(LAST_TRAIN - timedelta(days=89), LAST_TRAIN))
            .fillna(0))
    p = final_all[sku]
    fcast_s = pd.Series(p, index=FORECAST_DATES)

    ax.plot(hist.index, hist.values, color="steelblue", label="History (90d)")
    ax.plot(fcast_s.index[:28], fcast_s.values[:28],
            color="orange", linewidth=2, label="Validation (F1-28)")
    ax.plot(fcast_s.index[28:], fcast_s.values[28:],
            color="green", linewidth=2, label="Evaluation (F29-56)")
    ax.axvline(LAST_TRAIN, color="red", linestyle="--", alpha=0.7)
    ax.set_title(sku, fontsize=11)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))

plt.suptitle("Forecast vs History — Top 6 SKUs by Sales", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "top_sku_forecasts.png",
            dpi=120, bbox_inches="tight")
plt.show()
print("Saved top_sku_forecasts.png")

Saved top_sku_forecasts.png


## 19. Summary & Next Steps

| Metric | Value |
|--------|-------|
| Val WRMSSE | *(see cell 12 output)* |
| Test WRMSSE | *(see cell 12 output)* |
| Tier-3 SKUs (LightGBM+XGBoost) | ~5,000 |
| Tier-2 SKUs (Croston + seasonal) | ~3,000 |
| Tier-1 SKUs (recent mean) | ~3,500 |
| Tier-0 SKUs (predict 0) | ~4,000+ |

### Key improvements over v2
- **Return handling**: net daily qty (gross sales minus returns) instead of blind clip
- **WRMSSE metric**: properly scaled per M5 competition rules, with revenue-like weights
- **Time split**: Train / Val (28d) / Test (28d) — no leakage
- **Croston's method**: intermittent demand model for sparse SKUs (Tier 2)
- **XGBoost**: blended 40% with LightGBM for diversity
- **More features**: zero-rate, CV, YoY ratio, sin/cos calendar, Vietnamese holidays
- **Zero-inflation**: suppress near-zero forecasts for very intermittent SKUs

### Potential further improvements
1. **Prophet / statsforecast** for top-50 SKUs (capture trend + Fourier seasonality)
2. **Quantile blending**: train 10th-percentile and 90th-percentile models, pick based on CV
3. **Category grouping**: aggregate forecasts at product-family level, then disaggregate
4. **External data**: fuel prices, weather, economic indicators for auto parts demand
5. **Fourier features** for weekly/annual seasonality in dense matrix
